<a href="https://colab.research.google.com/github/guebin/DL2026/blob/master/posts/11wk-1.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" style="text-align: left"></a>

# 1. 강의영상

{{<video https://youtu.be/playlist?list=PLQqh36zP38-w26UVLonYupRuwwsCVH9nU&si=3vWKfq0HQEY8ByU4>}}

# 2. Imports

In [ ]:
import torch
import torchvision
import matplotlib.pyplot as plt

In [ ]:
plt.rcParams['figure.figsize'] = (4.5,3.0)

# 3. 다항분류

## A. 범주형자료

`-` 범주형자료를 숫자로 어떻게 바꿀까?

- 실패/성공 $\to$ 0/1
- 숫자3그림/숫자7그림 $\to$ 0/1
- 강아지그림/고양이그림 $\to$ 0/1
- 강아지그림/고양이그림/토끼그림 $\to$ 0/1/2 ???

`-` 주입식 교육

- 잘못된방식: 강아지그림 = 0, 고양이그림 = 1, 토끼그림 = 2
- 올바른방식: 강아지그림 = [1,0,0], 고양이그림 = [0,1,0], 토끼그림 = [0,0,1] // <-- 이러한 방식을 원핫인코딩이라함

`-` 왜??

- 설명1: 강아지그림, 고양이그림, 토끼그림은 서열척도가 아니라 명목척도다. 그래서 범주를 0,1,2로 으로 숫자화하면 평균등의 의미가 없음 (사회조사분석사 2급 스타일)
- 설명2: 범주형은 원핫인코딩으로 해야함 ("30일만에 끝내는 실전머신러닝" 이런 책에 나오는 스타일)
- 설명3: 동전을 한번 던져서 나오는 결과는 베르누이분포, 즉 시행횟수=1인 이항분포 $B(1,p)$를 따름. 주사위를 한번 던져서 나오는 눈금의 숫자는 시행횟수가=1인 다항분포 $M(1,{\bf p})$를 따름. 시행횟수가=1인 다항분포의 실현값은 0,1 이고, 시행횟수가=1인 다항분포의 실현값은 [1,0,0], [0,1,0], [0,0,1] 이므로 당연히 $y_i$ 는 [1,0,0], [0,1,0], [0,0,1] 중 하나의 형태랄 가정하는게 바람직함.


## B. 실습: 3개의 클래스를 구분

`-` 데이터준비

In [ ]:
train_dataset = torchvision.datasets.MNIST(root='./data',train=True,download=True)
to_tensor = torchvision.transforms.ToTensor()
X3 = torch.stack([to_tensor(Xi) for Xi, yi in train_dataset if yi==3])
X7 = torch.stack([to_tensor(Xi) for Xi, yi in train_dataset if yi==7])
X9 = torch.stack([to_tensor(Xi) for Xi, yi in train_dataset if yi==9])
X = torch.concat([X3,X7,X9]).reshape(-1,1*28*28)
y = torch.tensor([0]*6131 + [1]*6265+ [2]*len(X9)) # 숫자3은0, 숫자7은1, 숫자9는2

In [ ]:
print(X.shape)
print(y.shape)

torch.Size([18345, 784])
torch.Size([18345])


In [ ]:
y # 이러한식으로 y를 만들면 안된다고 했음

tensor([0, 0, 0,  ..., 2, 2, 2])

In [ ]:
Y = torch.nn.functional.one_hot(y).float()
Y

tensor([[1., 0., 0.],
        [1., 0., 0.],
        [1., 0., 0.],
        ...,
        [0., 0., 1.],
        [0., 0., 1.],
        [0., 0., 1.]])

`-` 적합

In [ ]:
torch.manual_seed(0)
net = torch.nn.Sequential(
    torch.nn.Linear(784,32),
    torch.nn.ReLU(),
    torch.nn.Linear(32,3),
)
loss_fn = torch.nn.CrossEntropyLoss() # 이게 이름이 CEWithLogitsLoss 였다면 좋았을텐데.
optimizer = torch.optim.Adam(net.parameters())
for epoch in range(100):
    #step1
    Logits = net(X)
    #step2
    loss = loss_fn(Logits,Y)
    #step3
    loss.backward()
    #step4
    optimizer.step()
    optimizer.zero_grad()

In [ ]:
(net(X).argmax(axis=1) == y).float().mean()

tensor(0.9531)

`-` 그런데 사실...

In [ ]:
torch.manual_seed(0)
net = torch.nn.Sequential(
    torch.nn.Linear(784,32),
    torch.nn.ReLU(),
    torch.nn.Linear(32,3),
)
loss_fn = torch.nn.CrossEntropyLoss() # 이게 이름이 CEWithLogitsLoss 였다면 좋았을텐데.
optimizer = torch.optim.Adam(net.parameters())
for epoch in range(100):
    #step1
    Logits = net(X)
    #step2
    loss = loss_fn(Logits,y)
    #step3
    loss.backward()
    #step4
    optimizer.step()
    optimizer.zero_grad()

In [ ]:
(net(X).argmax(axis=1) == y).float().mean()

tensor(0.9531)

> 파이토치 착해요. 친절함. Y가 아니고 y를 넣어도 돌려줘요.

In [ ]:
net(X).softmax(axis=1)

tensor([[9.9819e-01, 2.7907e-04, 1.5294e-03],
        [9.9341e-01, 1.5609e-03, 5.0314e-03],
        [9.9979e-01, 1.5516e-04, 5.1536e-05],
        ...,
        [2.8185e-03, 1.4823e-01, 8.4895e-01],
        [4.1827e-03, 2.6862e-02, 9.6896e-01],
        [2.8057e-02, 1.3784e-01, 8.3410e-01]], grad_fn=<SoftmaxBackward0>)

## C. 결론

`-` 조금 헷갈리지만...

|task | 오차항의가정 | 마지막활성화함수| netout의 의미 | 손실함수(파이토치) | 손실함수(개념)
|:--:|:--:|:--:|:--:|:--:|:--:|
|이항분류 | 시행횟수=1인 이항분포 | Sigmoid | prob | `BCELoss` | BCE |
|이항분류 | 시행횟수=1인 이항분포 | None | logit | `BCEWithLogitsLoss` | NA |
|다항분류 | 시행횟수=1인 다항분포 | Softmax |  probs |  NA | CE |
|다항분류 | 시행횟수=1인 다항분포 | Noen | logits |  `CrossEntropyLoss` | NA |

`-` 나같으면 이렇게 만들었겠다..

|task | 오차항의가정 | 마지막활성화함수| netout의 의미 | 손실함수(파이토치) | 손실함수(개념)
|:--:|:--:|:--:|:--:|:--:|:--:|
|이항분류 | 시행횟수=1인 이항분포 | Sigmoid | prob | `BCELoss` | BCE |
|이항분류 | 시행횟수=1인 이항분포 | None | logit | `BCEWithLogitsLoss` | NA |
|다항분류 | 시행횟수=1인 다항분포 | Softmax |  probs |  `CELoss` | CE |
|다항분류 | 시행횟수=1인 다항분포 | Noen | logits |  `CEWithLogitsLoss` | NA |